In [1]:
!pip install pandas
!pip install numpy
!pip install scikit-learn

In [2]:
# Import the pandas library, which is the best tool for working with data tables
import pandas as pd

# Define the path to our data file
# This tells the code where to find the file you downloaded
file_path = 'data/tox21.csv'

# Use pandas to read the CSV file into a data table called a DataFrame
df_tox = pd.read_csv(file_path)

# --- Let's inspect the data ---

# Print the first 5 rows of the table to see what it looks like
print("First 5 rows of the Tox21 dataset:")
display(df_tox.head())

# Print a summary of the dataset to understand its structure
print("\nDataset Information:")
df_tox.info()

First 5 rows of the Tox21 dataset:


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7831 entries, 0 to 7830
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   NR-AR          7265 non-null   float64
 1   NR-AR-LBD      6758 non-null   float64
 2   NR-AhR         6549 non-null   float64
 3   NR-Aromatase   5821 non-null   float64
 4   NR-ER          6193 non-null   float64
 5   NR-ER-LBD      6955 non-null   float64
 6   NR-PPAR-gamma  6450 non-null   float64
 7   SR-ARE         5832 non-null   float64
 8   SR-ATAD5       7072 non-null   float64
 9   SR-HSE         6467 non-null   float64
 10  SR-MMP         5810 non-null   float64
 11  SR-p53         6774 non-null   float64
 12  mol_id         7831 non-null   object 
 13  smiles         7831 non-null   object 
dtypes: float64(12), object(2)
memory usage: 856.6+ KB


In [3]:
# --- Step 2: Clean the data and select a single target ---

# Let's choose 'NR-AR' as our target for this prediction model
target_column = 'NR-AR'
feature_column = 'smiles'

# Create a new, smaller table (DataFrame) with only the columns we need
df_clean = df_tox[[feature_column, target_column]].copy()

# Remove all rows from our new table where the target column has a missing value (NaN)
df_clean.dropna(subset=[target_column], inplace=True)

# Print the shape of our new, clean dataset to see how many rows are left
print(f"Original number of molecules: {len(df_tox)}")
print(f"Number of molecules after cleaning for '{target_column}': {len(df_clean)}")

# Display the first 5 rows of our clean dataset
print("\nFirst 5 rows of the clean dataset:")
display(df_clean.head())

Original number of molecules: 7831
Number of molecules after cleaning for 'NR-AR': 7265

First 5 rows of the clean dataset:


,smiles,NR-AR
0,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,0.0
1,CCN1C(=O)NC(c2ccccc2)C1=O,0.0
3,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,0.0
4,CC(O)(P(=O)(O)O)P(=O)(O)O,0.0
5,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C,0.0


In [4]:
# --- Step 3: Generate molecular features using RDKit ---

from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

# Define a function to generate features for a single molecule (given as a SMILES string)
def generate_features(smiles_string):
    # Create a molecule object from the SMILES string
    mol = Chem.MolFromSmiles(smiles_string)
    
    # Check if the molecule was created successfully
    if mol is None:
        # Return a list of NaNs (Not a Number) if the SMILES is invalid
        return [np.nan, np.nan, np.nan, np.nan]
    
    # Calculate descriptors (features)
    mol_wt = Descriptors.MolWt(mol)
    log_p = Descriptors.MolLogP(mol)
    h_donors = Descriptors.NumHDonors(mol)
    h_acceptors = Descriptors.NumHAcceptors(mol)
    
    return [mol_wt, log_p, h_donors, h_acceptors]

# Apply our function to every SMILES string in the 'smiles' column
# This creates a new DataFrame with our 4 features for each molecule
features_df = df_clean['smiles'].apply(lambda x: pd.Series(generate_features(x)))
features_df.columns = ['MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors']

# --- Combine the features with our original clean DataFrame ---

# First, reset the index of both DataFrames to ensure they align perfectly
df_clean.reset_index(drop=True, inplace=True)
features_df.reset_index(drop=True, inplace=True)

# Now, join the new features table with our clean table
final_df = pd.concat([df_clean, features_df], axis=1)

# Display the first 5 rows of our final dataset with features
print("Final dataset with numerical features:")
display(final_df.head())

Final dataset with numerical features:


,smiles,NR-AR,MolWt,LogP,NumHDonors,NumHAcceptors
0,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,0.0,258.324,1.34240,1.0,5.0
1,CCN1C(=O)NC(c2ccccc2)C1=O,0.0,204.229,1.29940,1.0,2.0
2,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,0.0,276.424,3.75244,1.0,2.0
3,CC(O)(P(=O)(O)O)P(=O)(O)O,0.0,206.027,-0.99220,5.0,3.0
4,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C,0.0,290.444,4.81720,0.0,4.0


In [5]:
# --- Step 4: Train the XGBoost Prediction Model ---

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Separate our features (X) and our target (y)
# X is the set of columns the model will learn from.
features = ['MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors']
X = final_df[features]

# y is the answer column the model will try to predict.
y = final_df['NR-AR']

# 2. Split the data into training and testing sets
# 80% of the data will be for training, 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# 3. Initialize and train the XGBoost model
# We use XGBClassifier because we are predicting a category (0 or 1).
model_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# The .fit() command is where the model learns from the training data.
print("\nTraining the XGBoost model...")
model_xgb.fit(X_train, y_train)
print("Training complete.")

# 4. Make predictions and evaluate the model
# Now, we ask the trained model to predict the outcome for the test data.
y_pred = model_xgb.predict(X_test)

# Compare the model's predictions (y_pred) to the actual answers (y_test).
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Accuracy on Test Data: {accuracy * 100:.2f}%")

Training data shape: (5812, 4)
Testing data shape: (1453, 4)

Training the XGBoost model...


C:\Users\Dell\anaconda3_new\Lib\site-packages\xgboost\training.py:183: UserWarning: [20:52:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete.

Model Accuracy on Test Data: 95.32%


In [6]:
# --- Step 5: Save the trained model to a file ---
model_xgb.save_model("toxicity_predictor.json")

print("Model saved successfully!")

Model saved successfully!
